# Customer Radar — mô hình thầy Qwen3 gán nhãn kho đánh giá thật

**Chuẩn bị (một lần):**
1. Trên máy: `cd backend && node scripts/teacher_io.js export` → ra `nlp_service/data/real/teacher_input.jsonl` và `taxonomy.json`.
2. Tạo Kaggle Dataset tên `customer-radar-real` chứa hai tệp đó (Add Data → Upload).
3. Notebook này: Accelerator = **GPU T4 x2**, Internet = **On**, Add Data = dataset vừa tạo.

**Chạy:** Run All. Khoảng 2–4 giờ cho ~4.000 đánh giá với Qwen3-8B (đổi `MODEL_ID` sang `Qwen/Qwen3-4B` nếu muốn nhanh gấp đôi).
Tắt giữa chừng thì chạy lại: notebook tự bỏ qua những dòng đã có trong `teacher_output.jsonl`.

**Xong:** tải `/kaggle/working/teacher_output.jsonl` về, rồi trên máy:
`node scripts/teacher_io.js import đường/dẫn/teacher_output.jsonl --model=Qwen/Qwen3-8B`

Nhãn thầy chỉ dùng để HUẤN LUYỆN (trọng số thấp hơn nhãn người), không ghi đè nhãn đang hiển thị trên web.
Văn bản đã được che thông tin cá nhân trước khi xuất.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"      # hoặc "Qwen/Qwen3-4B" để nhanh hơn
BATCH = 8
MAX_NEW_TOKENS = 320

import glob, json, os, re, time
inp = glob.glob("/kaggle/input/**/teacher_input.jsonl", recursive=True)[0]
tax = json.load(open(glob.glob("/kaggle/input/**/taxonomy.json", recursive=True)[0], encoding="utf-8"))
OUT = "/kaggle/working/teacher_output.jsonl"
rows = [json.loads(l) for l in open(inp, encoding="utf-8") if l.strip()]
done = set()
if os.path.exists(OUT):
    done = {json.loads(l)["id"] for l in open(OUT, encoding="utf-8") if l.strip()}
todo = [r for r in rows if r["id"] not in done]
print(f"{len(rows)} đánh giá, đã có {len(done)}, còn {len(todo)}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Đã nạp", MODEL_ID)

In [ ]:
TAXONOMY = "\n".join(
    f"- {c['key']} ({c['label']}): " + " | ".join(f"{x['key']} = {x['label']}" for x in c["causes"]) for c in tax
)
CATS = {c["key"]: {x["key"] for x in c["causes"]} for c in tax}

SYSTEM = f'''Bạn gán nhãn đánh giá khách hàng tiếng Việt trên sàn thương mại điện tử.
TAXONOMY (chỉ dùng đúng các mã này):
{TAXONOMY}

Quy tắc:
- Hư hỏng do vận chuyển thuộc Delivery.DamagedInTransit, không thuộc ProductQuality.
- Chỉ ghi vấn đề khi khách PHÀN NÀN về nó. Khen thuần thì issues = [].
- MỈA MAI: người Việt hay dùng lời khen để chê ("giao nhanh ghê, 15 ngày mới tới",
  "chất lượng tuyệt vời, dùng 2 hôm là hỏng", "cảm ơn shop đã gửi nhầm màu").
  Dấu hiệu: lời khen đi cùng sự việc tiêu cực, số sao thấp mà chữ khen, "ghê", "quá trời",
  "đúng là", 🙂👏, câu hỏi tu từ. Khi mỉa mai: sentiment theo Ý THẬT, issues theo vấn đề THẬT.
- Số sao chỉ là gợi ý: 1 sao mà khen thật lòng vẫn là Positive.
- Một câu có thể nêu NHIỀU vấn đề (kể cả nhiều nguyên nhân trong cùng một nhóm): liệt kê đủ.

Trả về DUY NHẤT một JSON:
{{"sentiment": "Positive|Negative|Neutral", "sarcasm": true|false,
  "issues": [{{"category": "<mã nhóm>", "cause": "<mã nguyên nhân>"}}],
  "reason": "<một câu ngắn>"}}'''

def build(r):
    user = json.dumps({"text": r["text"], "rating": r["rating"]}, ensure_ascii=False)
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    # Tắt chế độ "suy nghĩ" của Qwen3: chỉ cần JSON, nhanh hơn nhiều lần
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def parse(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        o = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    if o.get("sentiment") not in ("Positive", "Negative", "Neutral"):
        return None
    issues = []
    for x in o.get("issues") or []:
        c, z = (x or {}).get("category"), (x or {}).get("cause")
        if c in CATS:
            issues.append({"category": c, "cause": z if z in CATS[c] else None})
    first = issues[0] if issues else None
    return {"sentiment": o["sentiment"], "sarcasm": o.get("sarcasm") is True, "issues": issues,
            "category": first["category"] if first else "Other", "subCategory": first["cause"] if first else None,
            "reason": str(o.get("reason") or "")[:300]}

In [ ]:
started, bad = time.time(), 0
with open(OUT, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), BATCH):
        part = todo[i:i + BATCH]
        enc = tok([build(r) for r in part], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.eos_token_id)
        outs = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for r, o in zip(part, outs):
            lab = parse(o)
            if lab is None:
                bad += 1
                continue
            f.write(json.dumps({"id": r["id"], **lab}, ensure_ascii=False) + "\n")
        f.flush()
        n = i + len(part)
        if n % (BATCH * 10) == 0 or n == len(todo):
            el = time.time() - started
            print(f"{n}/{len(todo)} | {el/60:.1f} phút | còn ~{el/n*(len(todo)-n)/60:.0f} phút | hỏng {bad}", flush=True)
print("Xong. Dòng không đọc được JSON:", bad)

In [ ]:
# Kiểm tra nhanh: phân bố nhãn, số câu mỉa mai, và độ lệch với số sao
out = [json.loads(l) for l in open(OUT, encoding="utf-8") if l.strip()]
by_id = {r["id"]: r for r in rows}
from collections import Counter
print("Cảm xúc:", Counter(o["sentiment"] for o in out))
print("Mỉa mai:", sum(o["sarcasm"] for o in out), "/", len(out))
low_pos = [o for o in out if (by_id[o["id"]]["rating"] or 3) <= 2 and o["sentiment"] == "Positive"]
print("1–2 sao mà thầy gán Tích cực:", len(low_pos))
for o in [o for o in out if o["sarcasm"]][:8]:
    print("-", by_id[o["id"]]["rating"], "sao |", by_id[o["id"]]["text"][:100], "=>", o["sentiment"], o["issues"])